# LAM-MSAC-style MoE baseline — DeepSense6G beam predictionThis notebook trains a **paper-inspired baseline** based on *Large AI Model forMultimodal Integrated Sensing and Communication* (LAM-MSAC; IEEE Network, 2026,DOI: [10.1109/MNET.2026.3661589](https://doi.org/10.1109/MNET.2026.3661589)).It evaluates the resulting model on this repository's validation split andcompares it with the saved AMBER ablation results.## What is implemented- Modality-specific feature encoders, followed by temporal feature tokens.- A learned router, top-k expert selection, and normalized expert weighting.- Parallel Transformer-style attention/FFN experts, with attention across the  available modality tokens.- A KL load-balancing penalty on the mean router probabilities.- A 64-beam prediction head and the same Top-1 / DBA metrics used by AMBER.## Relationship to AMBERThe two systems share multimodal encoders and Transformer-based fusion, butthey are not the same model. AMBER uses masked modality-specific/fusion blocksand a modality-weight indicator; this baseline uses a learned sparse expertrouter and an expert load-balancing loss. The notebook compares their resultson the same saved validation set rather than treating the two architectures asequivalent.## Adaptations and limitsThe paper describes RGB, scalar power, point-cloud, and complex mmWave inputs.This project has RGB frames, **LiDAR BEV maps**, **processed real-valued radarmaps**, and GPS. Accordingly, the notebook uses ResNet feature encoders for theimage/maps and an MLP for GPS; it cannot reproduce the paper's raw point-cloud,scalar-power, or complex-RF encoders. Power measurements are labels here, notinputs. Temporal map features are pooled to one token per frame; GPS contributestwo time tokens. These choices make this an adapted baseline, not an exactreproduction. Its experts are independent cross-modal Transformer blocks overthe available tokens; they are learned routes, not manually assigned modalitysubsets.The paper does not fully specify several implementation hyperparameters. Thoseare exposed below. Keep the validation split and beam-label convention fixed,and report the actual settings alongside results. Validation is used forcheckpoint selection; adaptation is an optional secondary evaluation. Theunlabelled test split cannot provide accuracy metrics.The learning rate, weight decay, and 50-epoch StepLR interval follow thepaper's described final settings. Epoch count, batch size, routing top-k, andload-balance weight are explicit notebook defaults where the paper does notgive a reproducible value.**Before running locally:** complete preprocessing so`data/processed_amber/index/samples.csv` and its referenced tensors exist.**On Kaggle:** enable Internet so the first cell can clone the public project,and attach the preprocessed dataset with **+ Add Input**. A GPU is recommended;CPU execution will be substantially slower.

## 1. Configuration and environment

In [ ]:
from pathlib import Pathimport osimport jsonimport randomimport shutilimport subprocessimport sysimport timeimport numpy as npimport pandas as pdimport torchimport torch.nn as nnimport torch.nn.functional as Fimport torchvisionfrom torch.utils.data import DataLoaderREPO_URL = (    "https://github.com/SathishAdithiyaaSV/"    "MultiModal-Beam-Prediction.git")def has_project_code(path):    return (path / "amber" / "dataset.py").is_file()def find_project_root():    candidates = [Path.cwd(), *Path.cwd().parents]    configured = os.environ.get("PROJECT_ROOT")    if configured:        candidates.insert(0, Path(configured).expanduser())    if Path("/kaggle/working").is_dir():        candidates.insert(0, Path("/kaggle/working/project"))    for candidate in candidates:        if has_project_code(candidate):            return candidate.resolve()    if Path("/kaggle/working").is_dir():        project = Path("/kaggle/working/project")        result = subprocess.run(            ["git", "clone", "--depth", "1", REPO_URL, str(project)],            capture_output=True,            text=True,            env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},            check=False,        )        if result.returncode == 0 and has_project_code(project):            return project.resolve()        detail = (result.stderr or result.stdout).strip()[:500]        raise RuntimeError(            "Could not prepare the project code in /kaggle/working/project. "            "Enable Kaggle Internet or attach a repository checkout as input. "            f"git clone output: {detail}"        )    raise FileNotFoundError(        "Could not locate the repository root. Set PROJECT_ROOT to the "        "repository path or run this notebook from the repository."    )ROOT = find_project_root()if str(ROOT) not in sys.path:    sys.path.insert(0, str(ROOT))from amber.config import MODALITIES, ModelConfigfrom amber.dataset import AmberDatasetfrom amber.metrics import evaluatefrom amber.train import pick_device, to_devicedef find_index(base):    for candidate in (        base / "data" / "processed_amber" / "index",        base / "processed_amber" / "index",        base / "index" / "index",        base / "index",    ):        if (candidate / "samples.csv").is_file():            return candidate    return Nonedef walk_directories(base):    for directory, subdirectories, filenames in os.walk(base):        yield Path(directory), subdirectories, filenamesdef find_input_index(input_root):    if not input_root.is_dir():        return None    for directory, _subdirectories, filenames in walk_directories(input_root):        if "samples.csv" in filenames:            return directory    return Nonedef find_source(base, name):    for candidate in (base / name / name, base / name):        if candidate.is_dir() and any(candidate.glob("scenario*")):            return candidate    return NoneINDEX_DIR = find_index(ROOT)DATA_ROOT = ROOTif INDEX_DIR is None and Path("/kaggle/input").is_dir():    INDEX_SOURCE = find_input_index(Path("/kaggle/input"))    if INDEX_SOURCE is not None:        candidate_roots = [            INDEX_SOURCE.parent,            INDEX_SOURCE.parent.parent,            INDEX_SOURCE,        ]        source_dirs = {            name: next(                (found for found in                 (find_source(base, name) for base in candidate_roots)                 if found is not None),                None,            )            for name in ("development", "adaptation", "test")        }        missing_sources = {name for name, path in source_dirs.items()                           if path is None}        if missing_sources:            for directory, subdirectories, _filenames in walk_directories(                Path("/kaggle/input")            ):                if (directory.name in missing_sources                        and any(name.startswith("scenario")                                for name in subdirectories)):                    source_dirs[directory.name] = directory                    missing_sources.remove(directory.name)                    if not missing_sources:                        break        dataset_stage = Path("/kaggle/working/data/processed_amber")        INDEX_DIR = dataset_stage / "index"        dataset_stage.mkdir(parents=True, exist_ok=True)        shutil.copytree(INDEX_SOURCE, INDEX_DIR, dirs_exist_ok=True)        for name, source in source_dirs.items():            if source is None:                continue            link = dataset_stage / name            if link.is_symlink():                try:                    same_source = link.resolve(strict=True) == source.resolve()                except FileNotFoundError:                    same_source = False                if not same_source:                    link.unlink()            elif link.exists() and not link.is_symlink():                raise FileExistsError(                    f"Expected a dataset source symlink at {link}; refusing "                    "to replace an existing directory."                )            if not link.exists():                link.symlink_to(source)        DATA_ROOT = INDEX_DIR.parents[2]if INDEX_DIR is None or not (INDEX_DIR / "samples.csv").is_file():    input_hint = (        "Attach the preprocessed DeepSense6G dataset with + Add Input; "        "it must contain index/samples.csv and the referenced source tensors."        if Path("/kaggle/input").is_dir()        else "Run `bash preprocessing_amber/run_preprocessing.sh` from the "             "repository root first."    )    raise FileNotFoundError(        f"Could not find processed index under {ROOT} or /kaggle/input. "        f"{input_hint}"    )SEED = 2022EPOCHS = 10BATCH_SIZE = 8LEARNING_RATE = 2e-4WEIGHT_DECAY = 1e-6NUM_EXPERTS = 4ROUTING_TOP_K = 1BALANCE_WEIGHT = 1e-2NUM_WORKERS = 2DEVICE = pick_device("auto")OUTPUT_DIR = ROOT / "results" / "lam_msac_comparison"OUTPUT_DIR.mkdir(parents=True, exist_ok=True)random.seed(SEED)np.random.seed(SEED)torch.manual_seed(SEED)if torch.cuda.is_available():    torch.cuda.manual_seed_all(SEED)print(f"code root: {ROOT}")print(f"dataset root: {DATA_ROOT}")print(f"index: {INDEX_DIR}")print(f"device: {DEVICE}")if DEVICE.type == "cuda":    print(f"GPU: {torch.cuda.get_device_name(0)}")print(    f"epochs={EPOCHS}, batch={BATCH_SIZE}, experts={NUM_EXPERTS}, "    f"top-k={ROUTING_TOP_K}")

## 2. Load the repository's existing train/validation splits

In [ ]:
MODEL_CONFIG = ModelConfig(    window=5,    n_beams=64,    pretrained=False,)SENSING_MODALITIES = ("image", "lidar", "radar", "gps")train_data = AmberDataset(    INDEX_DIR, "train", MODEL_CONFIG, root=DATA_ROOT,    modalities=SENSING_MODALITIES,)val_data = AmberDataset(    INDEX_DIR, "val", MODEL_CONFIG, root=DATA_ROOT,    modalities=SENSING_MODALITIES,)train_loader = DataLoader(    train_data, batch_size=BATCH_SIZE, shuffle=True,    num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",)val_loader = DataLoader(    val_data, batch_size=BATCH_SIZE, shuffle=False,    num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",)print(f"train={len(train_data):,}; validation={len(val_data):,}")print("train scenarios:", sorted(set(train_data.scenarios)))print("validation scenarios:", sorted(set(val_data.scenarios)))

## 3. Modality encoders and sparse MoE fusion

In [ ]:
class GridFrameEncoder(nn.Module):    def __init__(self, backbone, in_channels, embed_dim):        super().__init__()        net = backbone(weights=None)        if in_channels != 3:            old = net.conv1            net.conv1 = nn.Conv2d(                in_channels, old.out_channels, kernel_size=old.kernel_size,                stride=old.stride, padding=old.padding, bias=False,            )        self.trunk = nn.Sequential(*list(net.children())[:-2])        self.pool = nn.AdaptiveAvgPool2d(1)        self.proj = nn.Linear(net.fc.in_features, embed_dim)    def forward(self, frames):        batch, steps, channels, height, width = frames.shape        x = self.trunk(frames.reshape(batch * steps, channels, height, width))        x = self.pool(x).flatten(1)        return self.proj(x).reshape(batch, steps, -1)class CrossModalExpert(nn.Module):    def __init__(self, embed_dim, num_heads, dropout):        super().__init__()        self.attn_norm = nn.LayerNorm(embed_dim)        self.attn = nn.MultiheadAttention(            embed_dim, num_heads, dropout=dropout, batch_first=True,        )        self.ffn_norm = nn.LayerNorm(embed_dim)        self.ffn = nn.Sequential(            nn.Linear(embed_dim, 4 * embed_dim),            nn.GELU(),            nn.Dropout(dropout),            nn.Linear(4 * embed_dim, embed_dim),        )        self.dropout = nn.Dropout(dropout)    def forward(self, tokens, key_padding_mask):        q = self.attn_norm(tokens)        attended, _ = self.attn(            q, q, q, key_padding_mask=key_padding_mask, need_weights=False,        )        tokens = tokens + self.dropout(attended)        return tokens + self.dropout(self.ffn(self.ffn_norm(tokens)))class LAMMSACBeamPredictor(nn.Module):    def __init__(self, num_experts=4, top_k=1, embed_dim=256, num_heads=8,                 dropout=0.1):        super().__init__()        if not 1 <= top_k <= num_experts:            raise ValueError("top_k must be between 1 and num_experts")        self.num_experts = num_experts        self.top_k = top_k        self.modality_names = ("image", "lidar", "radar", "gps")        self.modality_indices = {            name: MODALITIES.index(name) for name in self.modality_names        }        self.encoders = nn.ModuleDict({            "image": GridFrameEncoder(torchvision.models.resnet50, 3, embed_dim),            "lidar": GridFrameEncoder(torchvision.models.resnet18, 1, embed_dim),            "radar": GridFrameEncoder(torchvision.models.resnet18, 2, embed_dim),            "gps": nn.Sequential(                nn.Linear(2, 64), nn.ReLU(),                nn.Linear(64, embed_dim), nn.LayerNorm(embed_dim),            ),        })        self.time_embeddings = nn.ParameterDict({            "image": nn.Parameter(torch.empty(1, 5, embed_dim)),            "lidar": nn.Parameter(torch.empty(1, 5, embed_dim)),            "radar": nn.Parameter(torch.empty(1, 5, embed_dim)),            "gps": nn.Parameter(torch.empty(1, 2, embed_dim)),        })        self.router = nn.Sequential(            nn.Linear(len(self.modality_names) * embed_dim + len(self.modality_names),                      embed_dim),            nn.GELU(),            nn.Linear(embed_dim, num_experts),        )        self.experts = nn.ModuleList([            CrossModalExpert(embed_dim, num_heads, dropout)            for _ in range(num_experts)        ])        self.output_norm = nn.LayerNorm(embed_dim)        self.readout_query = nn.Parameter(torch.empty(1, 1, embed_dim))        self.readout = nn.MultiheadAttention(            embed_dim, num_heads, dropout=dropout, batch_first=True,        )        self.classifier = nn.Linear(embed_dim, 64)        self.dropout = nn.Dropout(dropout)        self._initialize_parameters()    def _initialize_parameters(self):        for module in self.modules():            if isinstance(module, (nn.Linear, nn.Conv2d)):                nn.init.xavier_uniform_(module.weight)                if module.bias is not None:                    nn.init.zeros_(module.bias)            elif isinstance(module, nn.LayerNorm):                nn.init.ones_(module.weight)                nn.init.zeros_(module.bias)        for parameter in self.time_embeddings.values():            nn.init.normal_(parameter, std=0.02)        nn.init.normal_(self.readout_query, std=0.02)    def forward(self, batch):        availability = batch["availability"]        modality_tokens = {}        router_inputs = []        token_masks = []        for name in self.modality_names:            available = availability[:, self.modality_indices[name]].bool()            if name == "gps":                tokens = self.encoders[name](batch[name])            else:                tokens = self.encoders[name](batch[name])            tokens = tokens + self.time_embeddings[name]            tokens = tokens * available[:, None, None].to(tokens.dtype)            modality_tokens[name] = tokens            router_inputs.append(tokens.mean(dim=1))            token_masks.append(                available[:, None].expand(-1, tokens.shape[1])            )        pooled = torch.cat(router_inputs, dim=-1)        modality_available = torch.stack(            [availability[:, self.modality_indices[name]]             for name in self.modality_names],            dim=-1,        )        gate_input = torch.cat([pooled, modality_available], dim=-1)        router_probs = self.router(gate_input).softmax(dim=-1)        selected_probs, selected_experts = router_probs.topk(self.top_k, dim=-1)        selected_weights = selected_probs / selected_probs.sum(dim=-1, keepdim=True)        tokens = torch.cat(            [modality_tokens[name] for name in self.modality_names], dim=1,        )        token_available = torch.cat(token_masks, dim=1)        # Keep one harmless token unmasked for samples with no sensor flags.        empty_rows = ~token_available.any(dim=1)        if empty_rows.any():            token_available = token_available.clone()            token_available[empty_rows, 0] = True        key_padding_mask = ~token_available        fused = torch.zeros_like(tokens)        for expert_index, expert in enumerate(self.experts):            assigned = selected_experts == expert_index            rows = assigned.any(dim=1).nonzero(as_tuple=True)[0]            if rows.numel() == 0:                continue            rank = assigned[rows].to(torch.int64).argmax(dim=1)            weights = selected_weights[rows, rank]            expert_output = expert(tokens[rows], key_padding_mask[rows])            fused.index_add_(                0, rows, expert_output * weights[:, None, None],            )        fused = self.output_norm(fused)        query = self.readout_query.expand(fused.shape[0], -1, -1)        pooled_output, _ = self.readout(            query, fused, fused, key_padding_mask=key_padding_mask,            need_weights=False,        )        logits = self.classifier(self.dropout(pooled_output[:, 0]))        # KL(mean routing probability || uniform expert usage).        mean_probs = router_probs.mean(dim=0).clamp_min(1e-8)        balance_loss = (mean_probs * (mean_probs * self.num_experts).log()).sum()        return logits, balance_loss, router_probsmodel = LAMMSACBeamPredictor(    num_experts=NUM_EXPERTS, top_k=ROUTING_TOP_K,).to(DEVICE)print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Train with cross-entropy and router load balancing

In [ ]:
optimizer = torch.optim.AdamW(    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,)scheduler = torch.optim.lr_scheduler.StepLR(    optimizer, step_size=50, gamma=0.1,)@torch.no_grad()def evaluate_loader(loader):    model.eval()    all_logits, all_targets = [], []    for batch in loader:        batch = to_device(batch, DEVICE)        logits, _, _ = model(batch)        labelled = batch["target"] >= 0        if labelled.any():            all_logits.append(logits[labelled].float().cpu())            all_targets.append(batch["target"][labelled].cpu())    if not all_targets:        raise ValueError("Evaluation split contains no labelled examples")    return evaluate(torch.cat(all_logits), torch.cat(all_targets))history = []best_val_top1 = -1.0best_checkpoint = OUTPUT_DIR / "best_lam_msac.pt"for epoch in range(1, EPOCHS + 1):    model.train()    loss_sum = 0.0    labelled_count = 0    start = time.time()    for batch in train_loader:        batch = to_device(batch, DEVICE)        labelled = batch["target"] >= 0        if not labelled.any():            continue        batch = {key: value[labelled] for key, value in batch.items()}        logits, balance_loss, _ = model(batch)        loss = F.cross_entropy(logits, batch["target"])        total_loss = loss + BALANCE_WEIGHT * balance_loss        optimizer.zero_grad(set_to_none=True)        total_loss.backward()        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)        optimizer.step()        size = int(batch["target"].shape[0])        loss_sum += total_loss.detach().item() * size        labelled_count += size    scheduler.step()    val_metrics = evaluate_loader(val_loader)    row = {        "epoch": epoch,        "train_loss": loss_sum / max(labelled_count, 1),        "val_top1": val_metrics["top1"],        "val_top3": val_metrics["top3"],        "val_dba": val_metrics["dba"],        "seconds": time.time() - start,    }    history.append(row)    print(        f"epoch {epoch:03d}/{EPOCHS} "        f"loss={row['train_loss']:.4f} "        f"val_top1={row['val_top1']:.4f} "        f"val_dba={row['val_dba']:.4f} "        f"({row['seconds']:.1f}s)"    )    if row["val_top1"] > best_val_top1:        best_val_top1 = row["val_top1"]        torch.save(            {                "model": model.state_dict(),                "epoch": epoch,                "val_top1": best_val_top1,                "config": {                    "seed": SEED,                    "epochs": EPOCHS,                    "batch_size": BATCH_SIZE,                    "num_experts": NUM_EXPERTS,                    "routing_top_k": ROUTING_TOP_K,                    "balance_weight": BALANCE_WEIGHT,                    "modalities": list(SENSING_MODALITIES),                },            },            best_checkpoint,        )    pd.DataFrame(history).to_csv(OUTPUT_DIR / "history.csv", index=False)print(f"best validation Top-1={best_val_top1:.4f}; checkpoint={best_checkpoint}")

## 5. Restore the selected model and score validation

In [ ]:
checkpoint = torch.load(best_checkpoint, map_location=DEVICE, weights_only=False)model.load_state_dict(checkpoint["model"])lam_metrics = evaluate_loader(val_loader)lam_row = {    "model": "LAM-MSAC-style MoE (adapted)",    "Top-1": lam_metrics["top1"],    "Top-3": lam_metrics["top3"],    "DBA": lam_metrics["dba"],    "n": lam_metrics["n"],    "best_epoch": checkpoint["epoch"],    "parameters": sum(p.numel() for p in model.parameters()),}print(json.dumps(lam_row, indent=2))comparison_rows = [lam_row]amber_results = ROOT / "results" / "modality_ablation" / "results_all8.csv"if amber_results.is_file():    amber = pd.read_csv(amber_results)    for config_name in ("GPS + Radar + Camera + LiDAR", "GPS + Camera"):        row = amber.loc[amber["Configuration"] == config_name]        if not row.empty:            baseline = row.iloc[0]            comparison_rows.append({                "model": f"AMBER — {config_name}",                "Top-1": float(baseline["Top-1"]),                "Top-3": float(baseline["Top-3"]),                "DBA": float(baseline["DBA"]),                "n": int(baseline["n"]),                "best_epoch": int(baseline["Best epoch"]),                "parameters": int(float(baseline["Params (M)"]) * 1_000_000),            })else:    print(f"AMBER comparison CSV not found: {amber_results}")comparison = pd.DataFrame(comparison_rows)display(comparison)comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)with open(OUTPUT_DIR / "lam_msac_metrics.json", "w") as file:    json.dump(        {"validation": lam_metrics, "best_epoch": checkpoint["epoch"],         "config": checkpoint["config"]},        file, indent=2,    )

## 6. Optional labelled adaptation-split check

In [ ]:
try:    adaptation_data = AmberDataset(        INDEX_DIR, "adaptation", MODEL_CONFIG, root=DATA_ROOT,        modalities=SENSING_MODALITIES,    )except ValueError as exc:    print(f"Adaptation split unavailable: {exc}")else:    adaptation_loader = DataLoader(        adaptation_data, batch_size=BATCH_SIZE, shuffle=False,        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",    )    adaptation_metrics = evaluate_loader(adaptation_loader)    print("Adaptation is a secondary check, not used for checkpoint selection:")    print(json.dumps(adaptation_metrics, indent=2))    with open(OUTPUT_DIR / "adaptation_metrics.json", "w") as file:        json.dump(adaptation_metrics, file, indent=2)

## 7. Interpretation checklist- Compare the MoE model first with **AMBER all modalities** on the same  validation split. Both saved runs used 10 epochs, but differ in batch size,  optimizer settings, pretraining, and loss; treat this as an empirical  comparison, not a controlled architecture-only comparison.- The **GPS + Camera** AMBER result is a useful modality-efficiency reference,  not a same-input architecture comparison.- This notebook reports model parameters but does not measure sensor acquisition  cost, energy, or end-to-end hardware latency. Do not infer those costs from  parameter count.- One seed and one split do not establish statistical superiority. Repeat with  multiple seeds and retain the same split protocol before drawing conclusions.- This adapted implementation is not an exact reproduction of the paper's  sensor encoders or its Scenario 31/33-only study.